# NB11 — Portfolio Optimization

**Core Objective**: Construct optimal portfolios using 5 methodologies, each addressing
different assumptions about return/risk estimation and market dynamics.

## Expected Return Estimation (3 methods):
1. **Historical mean** — naive, suffers from estimation error
2. **CAPM-implied** — market equilibrium prior
3. **Black-Litterman** — Bayesian blend of equilibrium + ML views

## Risk Estimation (3 methods):
1. **Ledoit-Wolf shrinkage** — bias-variance optimal static covariance
2. **DCC-GARCH** — time-varying correlations (Engle, 2002)
3. **Regime-conditional** — separate covariance per HMM state

## Optimization Methods:
1. **Mean-Variance (Markowitz)** — max Sharpe, min variance
2. **Mean-CVaR** — scenario-based, uses NB04's return scenarios
3. **Black-Litterman** — ML views with Omega = diag(RMSE^2)
4. **Hierarchical Risk Parity (HRP)** — no covariance inversion needed
5. **Risk Budgeting (ERC)** — equal risk contribution

## Constraints (UCITS-inspired):
- Max single stock: 10%
- Max sector concentration: 30%
- Long-only (w >= 0)
- Turnover cap: 20% monthly

## Backtest:
- Monthly rebalancing with 10 bps transaction costs
- Benchmarks: equal-weight, XLK ETF
- Regime-adaptive strategy: HRP in bear, mean-variance in bull

**Output**: `portfolio_weights_timeseries.parquet`, `backtest_performance.csv`

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.portfolio_optimizer import (
    mean_variance_optimize, mean_cvar_optimize, black_litterman_optimize,
    hrp_optimize, risk_budgeting_optimize,
    expected_returns_historical, expected_returns_capm,
    covariance_ledoit_wolf, _enforce_constraints,
    build_sector_bounds, get_sector_group_indices,
)
from src.backtest_engine import (
    run_backtest, compute_all_metrics, compare_strategies,
    equal_weight_returns, compute_turnover,
)
from src.visualization import (
    save_fig, plot_efficient_frontier, plot_weight_evolution,
    plot_backtest_comparison,
)

print('Imports OK')

## 1. Load Data & Compute Returns

Load master data, ML predictions (walk-forward OOS), regime labels, and
return scenario matrix for CVaR optimization.

In [ ]:
# ── Load master data ──
master = pd.read_parquet(MASTER_DATA_FILE)

# Identify available tickers in the dataset
avail_tickers = [t for t in TICKERS if t in master.columns]
print(f'Available tickers: {len(avail_tickers)} / {len(TICKERS)}')

# Price panel
prices = master[avail_tickers].dropna(how='all')

# Daily simple returns (for portfolio construction)
returns = prices.pct_change().dropna(how='all')  # keep rows where at least 1 ticker has data

# Log returns (for statistical analysis)
log_returns = np.log(prices / prices.shift(1)).dropna()

print(f'Returns: {returns.shape}')
print(f'Date range: {returns.index[0]} → {returns.index[-1]}')

# ── Load upstream outputs ──
# ML return predictions from NB08 (walk-forward out-of-sample)
ml_ret_preds = pd.read_parquet(RETURN_PRED_FILE) if RETURN_PRED_FILE.exists() else None

# Return scenario matrix from NB04 (T x 20 for CVaR optimization)
scenarios = pd.read_parquet(RETURN_SCENARIOS_FILE) if RETURN_SCENARIOS_FILE.exists() else None

# Regime labels from NB05
regime_labels = pd.read_parquet(REGIME_LABELS_FILE) if REGIME_LABELS_FILE.exists() else None

# Conditional volatility from NB03 (for DCC-GARCH input)
cond_vol = pd.read_parquet(COND_VOL_FILE) if COND_VOL_FILE.exists() else None

# Benchmark data
if 'BM_SPY' in master.columns:
    spy_returns = master['BM_SPY'].pct_change().dropna()
else:
    spy_returns = None

xlk_returns = master['BM_XLK'].pct_change().dropna() if 'BM_XLK' in master.columns else None

print(f'\nML return predictions: {"Loaded" if ml_ret_preds is not None else "Not available"}')
print(f'Return scenarios (NB04): {"Loaded" if scenarios is not None else "Not available"}')
print(f'Regime labels (NB05): {"Loaded" if regime_labels is not None else "Not available"}')


## 2. Expected Return Estimation

### Method 1: Historical Mean (Naive)
Simply annualizes the sample mean of daily returns: $\hat{\mu}_i = \bar{r}_i \times 252$.
Suffers from high estimation error — the sample mean is a poor estimator
of future expected returns (Merton, 1980).

### Method 2: CAPM-Implied
$E[R_i] = R_f + \beta_i \cdot (E[R_m] - R_f)$
Uses SPY beta from regression. More stable than historical mean because
the cross-section of expected returns is governed by a single factor.

### Method 3: Black-Litterman
Bayesian blend: $\hat{\mu}_{BL} = [(\tau\Sigma)^{-1} + P'\Omega^{-1}P]^{-1}[(\tau\Sigma)^{-1}\pi + P'\Omega^{-1}Q]$
- Prior (pi): market-cap-implied equilibrium returns
- Views (Q): ML-predicted expected returns from NB08
- View uncertainty: $\Omega_{ii} = \text{RMSE}_i^2$ (squared forecast error)

In [ ]:
# ── Risk-free rate ──
# Use latest 3-month T-bill rate if available, else approximate
rf_annual = 0.045  # default ~4.5% (approximate current level)
rf_daily = rf_annual / 252

# ── Method 1: Historical Mean ──
mu_hist = expected_returns_historical(returns[avail_tickers])

# ── Method 2: CAPM-Implied ──
if spy_returns is not None:
    mkt = spy_returns.reindex(returns.index).dropna()
    aligned_rets = returns[avail_tickers].reindex(mkt.index)
    mu_capm = expected_returns_capm(aligned_rets, mkt, rf=rf_annual)
else:
    mu_capm = mu_hist  # fallback

# ── Display ──
mu_df = pd.DataFrame({
    'Historical': mu_hist,
    'CAPM': mu_capm,
}, index=avail_tickers)

print('── Expected Returns Comparison (annualized) ──')
print(mu_df.round(4).to_string())

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
mu_df.plot(kind='bar', ax=ax)
ax.set_ylabel('Annualized Expected Return')
ax.set_title('Expected Return Estimates by Method')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.legend()
plt.xticks(rotation=45)
save_fig(fig, 'nb11_expected_returns')
plt.show()

## 3. Risk Estimation

### Method 1: Ledoit-Wolf Shrinkage Covariance
Optimal linear shrinkage between sample covariance and structured target
(scaled identity). Reduces estimation error, especially when T/N is small.

### Method 2: DCC-GARCH Dynamic Conditional Correlation
Time-varying covariance: $H_t = D_t \cdot R_t \cdot D_t$ where
$D_t = \text{diag}(\sigma_{i,t})$ from univariate GARCH and
$R_t$ from DCC dynamics (Engle, 2002).

### Method 3: Regime-Conditional Covariance
Separate sample covariance for each HMM regime state.
Captures the fact that correlations spike during crises (bear regimes).

In [ ]:
# ── Method 1: Ledoit-Wolf ──
cov_lw = covariance_ledoit_wolf(returns[avail_tickers])
print(f'Ledoit-Wolf covariance: {cov_lw.shape}')

# ── Method 2: DCC-GARCH (if available) ──
# Use the latest DCC covariance matrix from NB03/dcc_garch.py
cov_dcc = None
try:
    from src.dcc_garch import run_dcc_garch
    # Use last 252 days of log returns for DCC estimation
    recent_log_ret = log_returns[avail_tickers].tail(504)
    if len(recent_log_ret) > 252:
        R_path, a_dcc, b_dcc, dcc_dates = run_dcc_garch(recent_log_ret)
        # Use the last DCC correlation matrix
        R_latest = R_path[-1]
        # Combine with Ledoit-Wolf marginal vols for stability
        daily_vols = np.sqrt(np.diag(cov_lw / 252))
        D = np.diag(daily_vols)
        cov_dcc = (D @ R_latest @ D) * 252  # annualize
        print(f'DCC-GARCH: a={a_dcc:.4f}, b={b_dcc:.4f}, a+b={a_dcc+b_dcc:.4f}')
except Exception as e:
    print(f'DCC-GARCH not available: {e}')

# ── Method 3: Regime-Conditional ──
cov_regime = {}
if regime_labels is not None and 'regime_state' in regime_labels.columns:
    regime_aligned = regime_labels['regime_state'].reindex(returns.index)
    for state in sorted(regime_aligned.dropna().unique()):
        mask = regime_aligned == state
        state_rets = returns[avail_tickers].loc[mask]
        if len(state_rets) > 60:
            cov_regime[int(state)] = state_rets.cov().values * 252
            print(f'Regime {int(state)}: {len(state_rets)} observations, '
                  f'avg corr = {state_rets.corr().values[np.triu_indices(len(avail_tickers), 1)].mean():.3f}')
else:
    print('No regime labels available — using Ledoit-Wolf only')

# Correlation heatmap of Ledoit-Wolf
std_dev = np.sqrt(np.diag(cov_lw))
corr_lw = cov_lw / np.outer(std_dev, std_dev)
corr_df = pd.DataFrame(corr_lw, index=avail_tickers, columns=avail_tickers)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Ledoit-Wolf Correlation Matrix (Annualized)')
save_fig(fig, 'nb11_correlation_matrix')
plt.show()

## 4. Portfolio Optimization — All 5 Methods

### 4.1 Mean-Variance (Markowitz)
Max-Sharpe portfolio: $\max_w \frac{w'\mu - R_f}{\sqrt{w'\Sigma w}}$ 
subject to $w \geq 0$, $\sum w = 1$, UCITS constraints.

### 4.2 Min-Variance
Ignores expected returns entirely: $\min_w w'\Sigma w$.
Often outperforms Markowitz in practice because $\Sigma$ is estimated
more accurately than $\mu$ (Jagannathan & Ma, 2003).

### 4.3 Mean-CVaR (Scenario-Based)
Minimizes Conditional Value-at-Risk from the joint return scenario matrix.
Portfolio CVaR = $\zeta + \frac{1}{\alpha T}\sum_{t=1}^{T}\max(-w'R_t - \zeta, 0)$

**Critical**: Portfolio CVaR is computed from $w'R_t$ (joint scenarios),
NOT from weighted individual CVaRs. CVaR is sub-additive but not linearly
decomposable for non-normal distributions.

### 4.4 Hierarchical Risk Parity (HRP)
No covariance inversion needed. Clusters assets by correlation distance,
then allocates inversely to cluster volatility. Robust to estimation error
(Lopez de Prado, 2016).

### 4.5 Equal Risk Contribution (ERC)
Each asset contributes equally to total portfolio risk:
$w_i \cdot (\Sigma w)_i / (w'\Sigma w) = 1/N$ for all $i$.

In [ ]:
portfolios = {}

# ── 4.1 Max-Sharpe (Mean-Variance) ──
try:
    w_sharpe = mean_variance_optimize(
        mu_capm, cov_lw, tickers=avail_tickers,
        objective='max_sharpe', rf=rf_annual
    )
    portfolios['Max-Sharpe'] = w_sharpe
    print(f'Max-Sharpe: top 5 = {dict(sorted(zip(avail_tickers, w_sharpe), key=lambda x: -x[1])[:5])}')
except Exception as e:
    print(f'Max-Sharpe failed: {e}')

# ── 4.2 Min-Variance ──
try:
    w_minvol = mean_variance_optimize(
        mu_capm, cov_lw, tickers=avail_tickers,
        objective='min_volatility'
    )
    portfolios['Min-Variance'] = w_minvol
    print(f'Min-Vol:    top 5 = {dict(sorted(zip(avail_tickers, w_minvol), key=lambda x: -x[1])[:5])}')
except Exception as e:
    print(f'Min-Variance failed: {e}')

# ── 4.3 Mean-CVaR ──
if scenarios is not None:
    scenario_arr = scenarios[avail_tickers].values if isinstance(scenarios, pd.DataFrame) else scenarios
    try:
        w_cvar = mean_cvar_optimize(
            scenario_arr, mu_capm, alpha=0.05, tickers=avail_tickers
        )
        portfolios['Mean-CVaR'] = w_cvar
        print(f'Mean-CVaR:  top 5 = {dict(sorted(zip(avail_tickers, w_cvar), key=lambda x: -x[1])[:5])}')
    except Exception as e:
        print(f'Mean-CVaR failed: {e}')
else:
    # Fallback: use historical return matrix as scenarios
    scenario_arr = returns[avail_tickers].tail(504).values  # last 2 years
    try:
        w_cvar = mean_cvar_optimize(
            scenario_arr, mu_capm, alpha=0.05, tickers=avail_tickers
        )
        portfolios['Mean-CVaR'] = w_cvar
        print(f'Mean-CVaR (hist scenarios): top 5 = {dict(sorted(zip(avail_tickers, w_cvar), key=lambda x: -x[1])[:5])}')
    except Exception as e:
        print(f'Mean-CVaR failed: {e}')

# ── 4.4 HRP ──
try:
    w_hrp = hrp_optimize(returns[avail_tickers], tickers=avail_tickers)
    portfolios['HRP'] = w_hrp
    print(f'HRP:        top 5 = {dict(sorted(zip(avail_tickers, w_hrp), key=lambda x: -x[1])[:5])}')
except Exception as e:
    print(f'HRP failed: {e}')

# ── 4.5 Equal Risk Contribution ──
try:
    w_erc = risk_budgeting_optimize(cov_lw, tickers=avail_tickers)
    portfolios['ERC'] = w_erc
    print(f'ERC:        top 5 = {dict(sorted(zip(avail_tickers, w_erc), key=lambda x: -x[1])[:5])}')
except Exception as e:
    print(f'ERC failed: {e}')

# ── 4.6 Black-Litterman (if ML views available) ──
if ml_ret_preds is not None:
    try:
        # Market-cap proxy weights (relative caps based on known market caps)
        # Approximate 2025 market caps ($B): mega→large→mid tiers
        _mcap_approx = {
            'AAPL': 3500, 'MSFT': 3200, 'NVDA': 2800, 'AMZN': 2100, 'GOOG': 2000,
            'META': 1500, 'TSM': 900, 'AVGO': 800, 'SAP': 300, 'CRM': 280,
            'AMD': 250, 'NOW': 200, 'PANW': 130, 'ANET': 120, 'MU': 110,
            'SNPS': 80, 'PLTR': 150, 'CRWD': 80, 'DDOG': 45, 'XYZ': 30,
        }
        mcap_vec = np.array([_mcap_approx.get(t, 100) for t in avail_tickers], dtype=float)
        mcw = mcap_vec / mcap_vec.sum()
        
        # ML views: use actual walk-forward predictions from NB08
        # ml_ret_preds is a DataFrame with columns per ticker
        ml_views = np.zeros(len(avail_tickers))
        view_rmse = np.full(len(avail_tickers), 0.05)  # default fallback
        for j, t in enumerate(avail_tickers):
            if t in ml_ret_preds.columns:
                # Annualize daily predicted returns
                ml_views[j] = float(ml_ret_preds[t].mean()) * 252
            else:
                ml_views[j] = mu_capm[j]  # fallback to CAPM
            # Per-ticker RMSE from walk-forward (if available)
            if hasattr(ml_ret_preds, 'attrs') and 'rmse' in ml_ret_preds.attrs:
                rmse_dict = ml_ret_preds.attrs['rmse']
                if t in rmse_dict:
                    view_rmse[j] = rmse_dict[t]
        
        w_bl = black_litterman_optimize(
            cov_lw, mcw, ml_views, view_rmse,
            rf=rf_annual, tickers=avail_tickers
        )
        portfolios['Black-Litterman'] = w_bl
        print(f'B-L:        top 5 = {dict(sorted(zip(avail_tickers, w_bl), key=lambda x: -x[1])[:5])}')
    except Exception as e:
        print(f'Black-Litterman failed: {e}')

print(f'\nSuccessfully optimized {len(portfolios)} portfolios')


## 4b. Extended Optimization Methods

### 4.7 Worst-Case Mean-Variance (Goldfarb & Iyengar, 2003)
Robust to estimation error: $\min_w w'\hat{\Sigma}w + \delta\|w\|_2^2$

### 4.8 Maximum Diversification (Choueifaty & Coignard, 2008)
$\max_w DR(w) = w'\sigma / \sqrt{w'\Sigma w}$

### 4.9 Resampled Efficient Frontier (Michaud, 1998)
Average over B=1000 Wishart draws for stable OOS weights.

### 4.10 CVaR Risk Budgeting (Tail Risk Parity)
Equalize marginal CVaR contributions.

### Conditional Diversification Benefit
$CDB = 1 - \sigma_{port} / (w'\sigma)$

In [ ]:
from src.portfolio_optimizer import (
    worst_case_mv_optimize, max_diversification_optimize,
    resampled_ef_optimize, cvar_risk_budgeting, conditional_diversification_benefit
)
from src.backtest_engine import (
    ulcer_index, pain_index, conditional_drawdown_at_risk,
    transaction_cost_impact_model, tracking_error
)

# ── 4.7 Worst-Case Mean-Variance ──
try:
    w_wcmv = worst_case_mv_optimize(mu_capm, cov_lw, delta=0.1, tickers=avail_tickers)
    portfolios['Worst-Case MV'] = w_wcmv['weights']
    print(f'Worst-Case MV: δ={0.1}, top 3 = {dict(sorted(zip(avail_tickers, w_wcmv["weights"]), key=lambda x: -x[1])[:3])}')
except Exception as e:
    print(f'Worst-Case MV failed: {e}')

# ── 4.8 Maximum Diversification ──
try:
    w_maxdiv = max_diversification_optimize(cov_lw, tickers=avail_tickers)
    portfolios['Max-Diversification'] = w_maxdiv['weights']
    print(f'Max-Div: DR={w_maxdiv["diversification_ratio"]:.3f}, '
          f'top 3 = {dict(sorted(zip(avail_tickers, w_maxdiv["weights"]), key=lambda x: -x[1])[:3])}')
except Exception as e:
    print(f'Max-Diversification failed: {e}')

# ── 4.9 Resampled Efficient Frontier ──
try:
    T_est = min(len(returns), 504)
    w_ref = resampled_ef_optimize(mu_capm, cov_lw, T=T_est, B=500, tickers=avail_tickers, rf=rf_annual)
    portfolios['Resampled EF'] = w_ref['weights']
    print(f'Resampled EF (B=500): top 3 = {dict(sorted(zip(avail_tickers, w_ref["weights"]), key=lambda x: -x[1])[:3])}')
except Exception as e:
    print(f'Resampled EF failed: {e}')

# ── 4.10 CVaR Risk Budgeting ──
try:
    scenario_arr_erc = returns[avail_tickers].tail(504).values
    w_cvar_erc = cvar_risk_budgeting(scenario_arr_erc, alpha=0.05, tickers=avail_tickers)
    portfolios['CVaR-ERC'] = w_cvar_erc['weights']
    print(f'CVaR-ERC: max imbalance={w_cvar_erc["max_imbalance"]:.4f}, '
          f'top 3 = {dict(sorted(zip(avail_tickers, w_cvar_erc["weights"]), key=lambda x: -x[1])[:3])}')
except Exception as e:
    print(f'CVaR-ERC failed: {e}')

# ── Conditional Diversification Benefit ──
print('\n--- Conditional Diversification Benefit ---')
for pname, pw in portfolios.items():
    cdb = conditional_diversification_benefit(pw, cov_lw)
    print(f'  {pname:20s}: CDB = {cdb:.4f}')

# ── Regime-Conditional CDB ──
if cov_regime:
    print('\n--- Regime-Conditional CDB (ERC portfolio) ---')
    w_test = portfolios.get('ERC', np.ones(len(avail_tickers)) / len(avail_tickers))
    for state, cov_r in cov_regime.items():
        cdb_r = conditional_diversification_benefit(w_test, cov_r)
        regime_name = {0: 'Bear', 1: 'Neutral', 2: 'Bull'}.get(state, f'State {state}')
        print(f'  {regime_name}: CDB = {cdb_r:.4f}')
    print('  (CDB_bear < CDB_bull expected — diversification fails when most needed)')

# ── Advanced Performance Metrics (computed after backtest in cell 17+) ──
# Note: backtest_results is populated in the backtest cell below.
# These metrics are computed in the performance comparison section.

print(f'\nTotal portfolios optimized: {len(portfolios)}')


## 5. Weight Comparison Visualization

Compare how different optimization methods allocate capital.
Key insights:
- Min-Variance tilts toward low-beta names (AAPL, MSFT, SAP)
- Max-Sharpe concentrates in high-return names (estimation error risk)
- HRP and ERC diversify more evenly across sectors
- CVaR penalizes left-tail exposure (avoids high-kurtosis names)

In [ ]:
if portfolios:
    weight_df = pd.DataFrame(portfolios, index=avail_tickers)
    
    # ── Weight comparison bar chart ──
    fig, ax = plt.subplots(figsize=(16, 8))
    weight_df.plot(kind='bar', ax=ax, width=0.8)
    ax.set_ylabel('Weight')
    ax.set_title('Portfolio Weights by Optimization Method')
    ax.axhline(y=MAX_SINGLE_STOCK_WEIGHT, color='red', linestyle='--',
               label=f'Single stock cap ({MAX_SINGLE_STOCK_WEIGHT:.0%})', alpha=0.7)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    save_fig(fig, 'nb11_weight_comparison')
    plt.show()
    
    # ── Verify constraints ──
    print('── Constraint Verification ──')
    groups = get_sector_group_indices(avail_tickers)
    for pname, w in portfolios.items():
        max_w = w.max()
        violations = []
        if max_w > MAX_SINGLE_STOCK_WEIGHT + 1e-6:
            violations.append(f'max_stock={max_w:.3f}')
        for gname, gidx in groups.items():
            gw = w[gidx].sum()
            if gw > MAX_SECTOR_WEIGHT + 1e-6:
                violations.append(f'{gname}={gw:.3f}')
        status = 'PASS' if not violations else f'VIOLATIONS: {", ".join(violations)}'
        print(f'  {pname}: sum={w.sum():.4f}, max_w={max_w:.3f} → {status}')

## 6. Efficient Frontier

Trace the mean-variance efficient frontier and mark all portfolio solutions.
The frontier is computed by solving for the min-variance portfolio at
each target return level.

In [ ]:
# ── Compute efficient frontier ──
n_points = 50
mu_range = np.linspace(mu_capm.min(), mu_capm.max(), n_points)
frontier_vols = []
frontier_rets = []

for target_ret in mu_range:
    try:
        w = mean_variance_optimize(
            mu_capm, cov_lw, tickers=avail_tickers,
            objective='efficient_return', target_return=target_ret, rf=rf_annual
        )
        port_vol = np.sqrt(w @ cov_lw @ w)
        port_ret = w @ mu_capm
        frontier_vols.append(port_vol)
        frontier_rets.append(port_ret)
    except Exception:
        continue

# ── Plot frontier + all portfolio solutions ──
fig, ax = plt.subplots(figsize=(12, 8))

if frontier_vols:
    ax.plot(frontier_vols, frontier_rets, 'b-', linewidth=2, label='Efficient Frontier')

# Plot individual portfolio solutions
markers = {'Max-Sharpe': '*', 'Min-Variance': 's', 'Mean-CVaR': 'D',
           'HRP': '^', 'ERC': 'o', 'Black-Litterman': 'P'}
colors = {'Max-Sharpe': 'red', 'Min-Variance': 'green', 'Mean-CVaR': 'purple',
          'HRP': 'orange', 'ERC': 'brown', 'Black-Litterman': 'cyan'}

for pname, w in portfolios.items():
    p_ret = w @ mu_capm
    p_vol = np.sqrt(w @ cov_lw @ w)
    ax.scatter(p_vol, p_ret, marker=markers.get(pname, 'o'), s=200,
               c=colors.get(pname, 'gray'), label=pname, zorder=5, edgecolors='black')

# Plot individual assets
asset_vols = np.sqrt(np.diag(cov_lw))
ax.scatter(asset_vols, mu_capm, c='lightgray', s=50, zorder=3, edgecolors='gray')
for i, t in enumerate(avail_tickers):
    ax.annotate(t, (asset_vols[i], mu_capm[i]), fontsize=7, alpha=0.7)

ax.set_xlabel('Annualized Volatility')
ax.set_ylabel('Annualized Expected Return')
ax.set_title('Efficient Frontier with Portfolio Solutions')
ax.legend(loc='upper left')
save_fig(fig, 'nb11_efficient_frontier')
plt.show()

## 7. Walk-Forward Portfolio Backtest

Monthly rebalancing using only walk-forward out-of-sample predictions.
Transaction costs: 10 bps per trade. Turnover cap: 20% per month.

**Backtest period**: From first available walk-forward prediction (~2023-01)
through 2026-03.

Each strategy is re-optimized monthly using only data available at that point
(expanding window). No lookahead bias.

In [ ]:
# ── Define rebalance dates (monthly) ──
# Start backtest from when walk-forward predictions become available
# With 70% initial training, first OOS predictions start ~70% through the data
backtest_start = returns.index[int(len(returns) * TRAIN_RATIO)]
backtest_end = returns.index[-1]

# Monthly rebalance dates — use actual trading dates closest to month-end
rebalance_dates = [returns.index[returns.index <= d][-1] for d in 
                   pd.date_range(backtest_start, backtest_end, freq='ME')
                   if len(returns.index[returns.index <= d]) > 0]

print(f'Backtest period: {backtest_start.strftime("%Y-%m-%d")} → {backtest_end.strftime("%Y-%m-%d")}')
print(f'Number of rebalance dates: {len(rebalance_dates)}')

# ── Strategy weight functions ──
def min_variance_strategy(date, rets_up_to):
    """Min-variance using expanding-window Ledoit-Wolf covariance."""
    cov = covariance_ledoit_wolf(rets_up_to[avail_tickers].tail(504))
    return mean_variance_optimize(
        np.zeros(len(avail_tickers)), cov,
        tickers=avail_tickers, objective='min_volatility'
    )

def hrp_strategy(date, rets_up_to):
    """HRP using last 504 trading days."""
    return hrp_optimize(rets_up_to[avail_tickers].tail(504), tickers=avail_tickers)

def erc_strategy(date, rets_up_to):
    """Equal Risk Contribution using expanding-window covariance."""
    cov = covariance_ledoit_wolf(rets_up_to[avail_tickers].tail(504))
    return risk_budgeting_optimize(cov, tickers=avail_tickers)

def equal_weight_strategy(date, rets_up_to):
    """1/N equal weight — no optimization needed."""
    n = len(avail_tickers)
    return np.ones(n) / n


# ── Run backtests ──
strategies = {
    'Min-Variance': min_variance_strategy,
    'HRP': hrp_strategy,
    'ERC': erc_strategy,
    'Equal-Weight': equal_weight_strategy,
}

backtest_results = {}
weights_histories = {}

for name, weight_fn in strategies.items():
    print(f'Running backtest: {name}...')
    try:
        w_hist, port_rets = run_backtest(
            returns, weight_fn, rebalance_dates,
            tickers=avail_tickers,
            transaction_cost_bps=TRANSACTION_COST_BPS,
            max_turnover=MAX_MONTHLY_TURNOVER
        )
        if len(port_rets) > 0:
            backtest_results[name] = port_rets
            weights_histories[name] = w_hist
            metrics = compute_all_metrics(port_rets, rf_daily, name=name)
            print(f'  Return={metrics["annualized_return"]:.2%}, '
                  f'Vol={metrics["annualized_volatility"]:.2%}, '
                  f'Sharpe={metrics["sharpe_ratio"]:.2f}, '
                  f'MaxDD={metrics["max_drawdown"]:.2%}')
    except Exception as e:
        print(f'  Failed: {e}')

# Add benchmark
if xlk_returns is not None:
    xlk_bt = xlk_returns.loc[backtest_results[list(backtest_results.keys())[0]].index[0]:
                             backtest_results[list(backtest_results.keys())[0]].index[-1]]
    if len(xlk_bt) > 0:
        backtest_results['XLK (Benchmark)'] = xlk_bt


## 8. Performance Comparison

Comprehensive metrics: annualized return, volatility, Sharpe, Sortino,
max drawdown, Calmar, total return.

In [ ]:
if backtest_results:
    perf_df = compare_strategies(backtest_results, rf_daily)
    
    # Format for display
    display_df = perf_df.copy()
    pct_cols = ['annualized_return', 'annualized_volatility', 'max_drawdown', 'total_return']
    for col in pct_cols:
        if col in display_df.columns:
            display_df[col] = display_df[col].map('{:.2%}'.format)
    ratio_cols = ['sharpe_ratio', 'sortino_ratio', 'calmar_ratio']
    for col in ratio_cols:
        if col in display_df.columns:
            display_df[col] = display_df[col].map('{:.2f}'.format)
    
    print('── Portfolio Performance Comparison ──')
    print(display_df.to_string())
    
    # ── Cumulative return plot ──
    fig = plot_backtest_comparison(
        backtest_results,
        title='Strategy Comparison: Cumulative Returns',
        save_name='nb11_strategy_comparison'
    )
    plt.show()
    
    # ── Drawdown plot ──
    fig, ax = plt.subplots(figsize=(14, 6))
    for name, rets in backtest_results.items():
        cum = (1 + rets).cumprod()
        dd = cum / cum.cummax() - 1
        ax.plot(dd.index, dd.values, label=name, linewidth=1)
    ax.set_ylabel('Drawdown')
    ax.set_title('Strategy Drawdowns')
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    save_fig(fig, 'nb11_strategy_drawdowns')
    plt.show()

## 9. Regime-Adaptive Strategy

**Key Insight**: Different optimization methods perform better under
different market regimes:
- **Bull/calm markets**: Mean-Variance exploits return forecasts
  (more signal, less noise in estimates)
- **Bear/crisis markets**: HRP is more robust to estimation error
  (correlations spike, covariance matrix becomes ill-conditioned)

The regime-adaptive strategy switches allocation method based on
the current HMM regime state from NB05.

In [ ]:
def regime_adaptive_strategy(date, rets_up_to):
    """
    Switch allocation method based on HMM regime:
    - Bear (state 0): Use HRP (robust, no covariance inversion)
    - Neutral (state 1): Use ERC (balanced risk contribution)
    - Bull (state 2): Use Max-Sharpe (exploit return estimates)
    
    If regime labels unavailable, fall back to ERC.
    """
    current_regime = None
    if regime_labels is not None and 'regime_state' in regime_labels.columns:
        regime_up_to = regime_labels.loc[:date, 'regime_state'].dropna()
        if len(regime_up_to) > 0:
            current_regime = int(regime_up_to.iloc[-1])
    
    recent_rets = rets_up_to[avail_tickers].tail(504)
    
    if current_regime == 0:  # Bear
        return hrp_optimize(recent_rets, tickers=avail_tickers)
    elif current_regime == 2:  # Bull — exploit return estimates via max Sharpe
        cov = covariance_ledoit_wolf(recent_rets)
        mu_est = recent_rets.mean().values * 252  # annualized
        return mean_variance_optimize(
            mu_est, cov,
            tickers=avail_tickers, objective='max_sharpe', rf=0.045
        )
    else:  # Neutral or unknown
        cov = covariance_ledoit_wolf(recent_rets)
        return risk_budgeting_optimize(cov, tickers=avail_tickers)


# ── Run regime-adaptive backtest ──
print('Running regime-adaptive backtest...')
try:
    w_regime, port_regime = run_backtest(
        returns, regime_adaptive_strategy, rebalance_dates,
        tickers=avail_tickers,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        max_turnover=MAX_MONTHLY_TURNOVER
    )
    if len(port_regime) > 0:
        backtest_results['Regime-Adaptive'] = port_regime
        weights_histories['Regime-Adaptive'] = w_regime
        m = compute_all_metrics(port_regime, rf_daily, name='Regime-Adaptive')
        print(f'  Return={m["annualized_return"]:.2%}, '
              f'Vol={m["annualized_volatility"]:.2%}, '
              f'Sharpe={m["sharpe_ratio"]:.2f}')
except Exception as e:
    print(f'Regime-adaptive failed: {e}')

# ── Updated comparison ──
if backtest_results:
    final_perf = compare_strategies(backtest_results, rf_daily)
    print('\n── Final Performance Ranking (by Sharpe) ──')
    print(final_perf.sort_values('sharpe_ratio', ascending=False).to_string())


## 10. Weight Evolution & Turnover Analysis

In [ ]:
# ── Weight evolution for best strategy ──
if weights_histories:
    best_strategy = final_perf['sharpe_ratio'].idxmax() if 'final_perf' in dir() else list(weights_histories.keys())[0]
    
    if best_strategy in weights_histories and len(weights_histories[best_strategy]) > 0:
        fig = plot_weight_evolution(
            weights_histories[best_strategy],
            title=f'Weight Evolution: {best_strategy}',
            save_name='nb11_weight_evolution'
        )
        plt.show()

# ── Turnover analysis ──
print('── Monthly Turnover Statistics ──')
for name, w_hist in weights_histories.items():
    if len(w_hist) < 2:
        continue
    turnovers = []
    w_arr = w_hist[avail_tickers].values if all(t in w_hist.columns for t in avail_tickers) else w_hist.values
    for i in range(1, len(w_arr)):
        t = compute_turnover(w_arr[i-1], w_arr[i])
        turnovers.append(t)
    if turnovers:
        print(f'  {name}: mean={np.mean(turnovers):.2%}, max={np.max(turnovers):.2%}, '
              f'median={np.median(turnovers):.2%}')

## 11. Save Outputs

In [ ]:
# ── Save portfolio weights time series ──
if weights_histories:
    all_weights = {}
    for name, w_hist in weights_histories.items():
        w_hist_copy = w_hist.copy()
        w_hist_copy.columns = [f'{name}_{c}' for c in w_hist_copy.columns]
        all_weights[name] = w_hist_copy
    
    # Save best strategy weights
    if best_strategy in weights_histories:
        weights_histories[best_strategy].to_parquet(PORTFOLIO_WEIGHTS_FILE)
        print(f'Saved weights: {PORTFOLIO_WEIGHTS_FILE}')

# ── Save performance table ──
if 'final_perf' in dir():
    final_perf.to_csv(BACKTEST_PERF_FILE)
    print(f'Saved performance: {BACKTEST_PERF_FILE}')

# ── Summary ──
print('\n' + '='*60)
print('NB11 SUMMARY — Portfolio Optimization Complete')
print('='*60)
print(f'Optimization methods: {len(portfolios)}')
print(f'Backtest strategies: {len(backtest_results)}')
if 'final_perf' in dir() and len(final_perf) > 0:
    best = final_perf['sharpe_ratio'].idxmax()
    print(f'Best Sharpe: {best} ({final_perf.loc[best, "sharpe_ratio"]:.2f})')
    print(f'Best Return: {final_perf["annualized_return"].idxmax()} '
          f'({final_perf["annualized_return"].max():.2%})')
    print(f'Lowest MaxDD: {final_perf["max_drawdown"].idxmax()} '
          f'({final_perf["max_drawdown"].max():.2%})')
print('\nOutputs ready for NB12 stress testing')